In [ ]:
import pandas as pd, numpy as np, statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from itertools import combinations
from statsmodels.stats.multitest import multipletests
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import warnings; warnings.filterwarnings('ignore')

prices = pd.read_csv("data/sp500_190_16y_adjclose_ffill.csv", index_col=0, parse_dates=True)
tickers = list(prices.columns)
print(f"LOADED: data/sp500_190_16y_adjclose_ffill.csv")
print(f"{len(tickers)} tickers, {len(prices)} days")
print("Shape:", prices.shape)
print("First tickers:", tickers[:5])
prices.head()


LOADED: data/sp500_190_16y_adjclose_ffill.csv
190 tickers, 4088 days
Shape: (4088, 190)
First tickers: ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'GOOGL']


,NVDA,AAPL,MSFT,AMZN,GOOGL,GOOG,META,AVGO,TSLA,BRK-B,...,NSC,ELV,SRE,DLR,FTNT,SPG,O,PCAR,APO,AZO
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,0.423784,6.412385,23.077381,6.6950,15.555863,15.483124,NaN,1.325782,NaN,66.220001,...,36.246201,47.450974,16.792442,26.434183,1.800,36.858234,11.630356,13.978406,NaN,158.029999
2010-01-05,0.429972,6.423470,23.084835,6.7345,15.487363,15.414942,NaN,1.335623,NaN,66.540001,...,36.804482,47.967159,16.434961,26.608126,1.839,36.606049,11.826418,14.230408,NaN,156.710007
2010-01-06,0.432722,6.321296,22.943171,6.6125,15.096945,15.026350,NaN,1.346168,NaN,66.199997,...,36.701088,48.745438,16.209660,26.249693,1.941,36.195045,11.817515,14.417504,NaN,155.240005
2010-01-07,0.424242,6.309609,22.704563,6.5000,14.745495,14.676543,NaN,1.337732,NaN,66.459999,...,36.218636,50.556114,16.173615,26.481630,1.945,36.666748,12.067053,14.608410,NaN,157.300003
2010-01-08,0.425159,6.351558,22.861145,6.6760,14.942070,14.872198,NaN,1.347574,NaN,66.440002,...,37.466141,50.778488,16.164604,26.776793,2.011,35.788750,12.231916,14.627506,NaN,155.279999


In [4]:
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
import numpy as np
import pandas as pd

prices    = pd.read_csv('data/sp500_190_16y_adjclose_ffill.csv', index_col=0, parse_dates=True)
all_pairs = pd.read_csv('data/sp500_17955_pairs.csv')

MIN_DAYS = 756 

results = []

for _, row in tqdm(all_pairs.iterrows(), total=len(all_pairs)):
    s1, s2 = row['stock1'], row['stock2']

    if s1 not in prices.columns or s2 not in prices.columns:
        continue

    pairdata = prices[[s1, s2]].dropna()
    if len(pairdata) < MIN_DAYS:
        continue

    y     = np.log(pairdata[s1].values)
    x     = np.log(pairdata[s2].values)
    beta  = np.polyfit(x, y, 1)[0]
    resid = y - beta * x
    adf_p = adfuller(resid, maxlag=1, autolag=None)[1]

    results.append({'stock1': s1, 'stock2': s2, 'adf_p': adf_p, 'beta': beta})

results_df = pd.DataFrame(results)

reject, _, _, _ = multipletests(results_df['adf_p'], alpha=0.15, method='fdr_bh')
results_df['fdr_reject'] = reject
fdr_pairs = results_df[results_df['adf_p'] < 0.10].copy()

results_df.to_csv('results/engle_granger_full_results.csv', index=False)
fdr_pairs.to_csv('results/fdr_surviving_pairs.csv', index=False)

print(f"Pairs tested (3yr+ only): {len(results_df)}")
print(f"ADF < 0.10:               {len(fdr_pairs)}")
print(f"FDR α=0.15:               {len(results_df[results_df.fdr_reject])}")
print(fdr_pairs.head())



100%|██████████| 17955/17955 [00:12<00:00, 1427.28it/s]


Pairs tested (3yr+ only): 17578
ADF < 0.10:               5837
FDR α=0.15:               2066
   stock1 stock2     adf_p      beta  fdr_reject
0    NVDA   AAPL  0.099488  1.861423       False
6    NVDA   AVGO  0.059186  1.353230       False
8    NVDA  BRK-B  0.024552  3.569229       False
11   NVDA    JPM  0.025936  2.840534       False
16   NVDA   ORCL  0.045076  3.152595       False


In [5]:
from statsmodels.tsa.stattools import coint
from tqdm import tqdm
import numpy as np
import pandas as pd

prices    = pd.read_csv('data/sp500_190_16y_adjclose_ffill.csv', index_col=0, parse_dates=True)

fdr_pairs = pd.read_csv('results/fdr_surviving_pairs.csv')
fdr_pairs = fdr_pairs[fdr_pairs['fdr_reject'] == True].copy() 

WINDOW    = 252
STEP      = 20
THRESHOLD = 0.05
MIN_DAYS  = 756

rolling_results = []

for _, row in tqdm(fdr_pairs.iterrows(), total=len(fdr_pairs)):
    s1, s2 = row['stock1'], row['stock2']

    if s1 not in prices.columns or s2 not in prices.columns:
        continue

    pairdata = prices[[s1, s2]].dropna()
    if len(pairdata) < MIN_DAYS:
        continue

    count_stationary = 0
    total_windows    = 0

    for start in range(0, len(pairdata) - WINDOW, STEP):
        end  = start + WINDOW
        s1_  = np.log(pairdata[s1].iloc[start:end].values)
        s2_  = np.log(pairdata[s2].iloc[start:end].values)
        pval = coint(s1_, s2_)[1]
        if pval < THRESHOLD:
            count_stationary += 1
        total_windows += 1

    stability = count_stationary / total_windows if total_windows > 0 else 0
    rolling_results.append({'stock1': s1, 'stock2': s2, 'stability': stability})

rolling_df   = pd.DataFrame(rolling_results)
ranked_pairs = rolling_df.sort_values(by='stability', ascending=False)

rolling_df.to_csv('results/rolling_cointegration_results.csv', index=False)
ranked_pairs.to_csv('results/ranked_pairs.csv', index=False)
print(f"Total pairs tested : {len(rolling_df)}")
print(f"\nTop 20:")
print(ranked_pairs.head(20).to_string())



100%|██████████| 2066/2066 [26:51<00:00,  1.28it/s]   

Total pairs tested : 2066

Top 20:
     stock1 stock2  stability
69    GOOGL   GOOG   0.364583
376      MA   UBER   0.240000
1119   UBER    ECL   0.213333
1379    SYK   INTU   0.208333
1479   INTU   MRSH   0.203125
1138    HON    SHW   0.203125
1948   ABNB    APD   0.200000
989     TXN   SPGI   0.197917
958     CRM     CI   0.197917
1124    HON    SYK   0.197917
1566    MCK   DELL   0.192661
1104   SCHW    HLT   0.188811
968     TMO    HON   0.187500
760     LIN   QCOM   0.187500
239     JPM   ANET   0.182482
620     MRK    UNP   0.182292
400      MA    NSC   0.182292
863     NEE    TMO   0.182292
62     AMZN   DASH   0.181818
1115    APP    COF   0.180000


In [6]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import numpy as np

prices = pd.read_csv('data/sp500_190_16y_adjclose_ffill.csv', index_col=0, parse_dates=True)

ranked_pairs = pd.read_csv('results/ranked_pairs.csv')
top20 = list(zip(ranked_pairs['stock1'].head(20), ranked_pairs['stock2'].head(20)))


pair_colors = [
    ('#e6194b', '#3cb44b'), ('#4363d8', '#f58231'), ('#911eb4', '#42d4f4'),
    ('#f032e6', '#a0522d'), ('#469990', '#e6194b'), ('#3cb44b', '#4363d8'),
    ('#f58231', '#911eb4'), ('#42d4f4', '#f032e6'), ('#a0522d', '#469990'),
    ('#2ecc71', '#e74c3c'), ('#3498db', '#f39c12'), ('#9b59b6', '#1abc9c'),
    ('#e67e22', '#2980b9'), ('#27ae60', '#c0392b'), ('#16a085', '#d35400'),
    ('#8e44ad', '#2c3e50'), ('#c0392b', '#27ae60'), ('#f39c12', '#9b59b6'),
    ('#1abc9c', '#e67e22'), ('#2c3e50', '#16a085')
]

GREY = '#aaaaaa'
ROWS, COLS = 5, 4

fig = make_subplots(
    rows=ROWS, cols=COLS,
    subplot_titles=[f"{s1}  vs  {s2}" for s1, s2 in top20],
    vertical_spacing=0.08,
    horizontal_spacing=0.07
)

for i, (s1, s2) in enumerate(top20):
    row = i // COLS + 1
    col = i % COLS  + 1

    if s1 not in prices.columns or s2 not in prices.columns:
        continue

    p1 = prices[s1].dropna()
    p2 = prices[s2].dropna()

    if p1.index[0] <= p2.index[0]:
        older_t, younger_t = s1, s2
        older_p, younger_p = p1, p2
        older_col  = pair_colors[i][0]
        younger_col = pair_colors[i][1]
    else:
        older_t, younger_t = s2, s1
        older_p, younger_p = p2, p1
        older_col  = pair_colors[i][1]
        younger_col = pair_colors[i][0]

    younger_start = younger_p.index[0]
    overlap = older_p.index[older_p.index >= younger_start]
    if len(overlap) == 0:
        continue

    norm_date = overlap[0]
    ov = older_p.loc[norm_date]
    yv = younger_p.loc[norm_date] if norm_date in younger_p.index else younger_p.iloc[0]

    older_n  = (older_p  / ov)  * 100
    younger_n = (younger_p / yv) * 100

   
    pre = older_n[older_n.index < younger_start]
    if len(pre) > 0:
        fig.add_trace(go.Scatter(
            x=pre.index, y=pre.values, mode='lines',
            line=dict(color=GREY, width=1.5),
            showlegend=False,
            hovertemplate=f"{older_t} (pre-{younger_t})<br>%{{x|%Y}}: %{{y:.0f}}<extra></extra>"
        ), row=row, col=col)

   
    post_old = older_n[older_n.index >= younger_start]
    fig.add_trace(go.Scatter(
        x=post_old.index, y=post_old.values, mode='lines',
        line=dict(color=older_col, width=2),
        showlegend=False,
        hovertemplate=f"{older_t}<br>%{{x|%Y}}: %{{y:.0f}}<extra></extra>"
    ), row=row, col=col)


    fig.add_trace(go.Scatter(
        x=younger_n.index, y=younger_n.values, mode='lines',
        line=dict(color=younger_col, width=2),
        showlegend=False,
        hovertemplate=f"{younger_t}<br>%{{x|%Y}}: %{{y:.0f}}<extra></extra>"
    ), row=row, col=col)

fig.update_layout(
    height=1800,
    width=1600,
    title={"text": "Top 20 Pairs — Normalized Prices (Rebased=100 at Overlap Start)<br>"
                   "<span style='font-size:14px;font-weight:normal;'>"
                   "Grey = older stock pre-younger IPO · Two colors per pair post-overlap</span>"},
    showlegend=False,
    paper_bgcolor='#1e1e1e',
    plot_bgcolor='#1e1e1e',
    font=dict(color='white')
)

fig.update_xaxes(
    tickformat="%Y",
    dtick="M24",
    tickangle=45,
    showgrid=True,
    gridcolor='#444444',
    gridwidth=0.8,
    tickfont=dict(size=8),
    minor=dict(
        dtick="M12",
        showgrid=True,
        gridcolor='#2a2a2a',
        gridwidth=0.4
    )
)

fig.update_yaxes(
    showgrid=True,
    gridcolor='#444444',
    gridwidth=0.8,
    nticks=6,
    tickfont=dict(size=8)
)

fig.write_image('results/top20_pairs_plot.png', width=1600, height=1800)
print("Saved → results/top20_pairs_plot.png")


Saved → results/top20_pairs_plot.png


In [7]:
final_pairs = ranked_pairs.head(21)
final_pairs = final_pairs[~((final_pairs.stock1 == 'GOOGL') & (final_pairs.stock2 == 'GOOG'))]
final_pairs = final_pairs.head(20).reset_index(drop=True)

final_pairs.to_csv('./results/final_top20_pairs.csv', index=False)
print(f"Saved {len(final_pairs)} pairs to results/final_top20_pairs.csv")
print(final_pairs[['stock1','stock2','stability']].to_string())


Saved 20 pairs to results/final_top20_pairs.csv
   stock1 stock2  stability
0      MA   UBER   0.240000
1    UBER    ECL   0.213333
2     SYK   INTU   0.208333
3    INTU   MRSH   0.203125
4     HON    SHW   0.203125
5    ABNB    APD   0.200000
6     TXN   SPGI   0.197917
7     CRM     CI   0.197917
8     HON    SYK   0.197917
9     MCK   DELL   0.192661
10   SCHW    HLT   0.188811
11    TMO    HON   0.187500
12    LIN   QCOM   0.187500
13    JPM   ANET   0.182482
14    MRK    UNP   0.182292
15     MA    NSC   0.182292
16    NEE    TMO   0.182292
17   AMZN   DASH   0.181818
18    APP    COF   0.180000
19    TXN    MCO   0.177083


In [8]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path('results')


ranked_all = pd.read_csv(RESULTS_DIR / 'ranked_pairs.csv')


remove = (
    ((ranked_all.stock1 == 'GOOGL') & (ranked_all.stock2 == 'GOOG')) |
    ((ranked_all.stock1 == 'GOOG') & (ranked_all.stock2 == 'GOOGL'))
)
ranked_all = ranked_all[~remove].reset_index(drop=True)

final_pairs = ranked_all.head(19).copy().reset_index(drop=True)

final_pairs.to_csv(RESULTS_DIR / 'final_candidate_pairs.csv', index=False)

print(f"Final candidate pairs : {len(final_pairs)}")
print(f"Saved → results/final_candidate_pairs.csv")
final_pairs[['stock1', 'stock2', 'stability']]



Final candidate pairs : 19
Saved → results/final_candidate_pairs.csv


,stock1,stock2,stability
0,MA,UBER,0.240000
1,UBER,ECL,0.213333
2,SYK,INTU,0.208333
3,INTU,MRSH,0.203125
4,HON,SHW,0.203125
5,ABNB,APD,0.200000
6,TXN,SPGI,0.197917
7,CRM,CI,0.197917
8,HON,SYK,0.197917
9,MCK,DELL,0.192661


In [9]:
final_pairs[['stock1', 'stock2', 'stability']]

,stock1,stock2,stability
0,MA,UBER,0.240000
1,UBER,ECL,0.213333
2,SYK,INTU,0.208333
3,INTU,MRSH,0.203125
4,HON,SHW,0.203125
5,ABNB,APD,0.200000
6,TXN,SPGI,0.197917
7,CRM,CI,0.197917
8,HON,SYK,0.197917
9,MCK,DELL,0.192661
